# 1. Load the Data

We're going to load pictures of 31 different kinds of fish. We'll have 5 images per kind of fish in each of our training, testing, and validation sets. So 15 images per kind of fish. A very small dataset!

In [ ]:
import subprocess

In [ ]:
subprocess.run(
    ["curl", "-L", "-o", "/content/fish-dataset.zip", "https://www.kaggle.com/api/v1/datasets/download/markdaniellampa/fish-dataset"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

In [ ]:
subprocess.run(
    ["unzip", "/content/fish-dataset.zip"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

In [ ]:
from tqdm import tqdm
import shutil
import os

if os.path.exists('data'):
    shutil.rmtree('data')

if not os.path.exists('data'):
    os.mkdir('data')
    os.mkdir('data/train')
    os.mkdir('data/test')
    os.mkdir('data/val')

num_images = 5

for _case in ['train', 'test', 'val']:
    source_base = f'/content/FishImgDataset/{_case}'
    for fish in tqdm(os.listdir(source_base)):
        source_fish = os.path.join(source_base, fish)
        dest_fish = os.path.join('data', _case, fish)
        os.mkdir(dest_fish)
        paths = sorted(os.listdir(source_fish))
        for path in paths[:num_images]:
            source_path = os.path.join(source_fish, path)
            shutil.copy(source_path, dest_fish)

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_datagen = ImageDataGenerator(rescale = 1./255, #convert to target values between 0 and 1 for faster training
                                   shear_range = 0.2, #for randomly applying shearing transformations
                                   zoom_range = 0.2,  # for randomly zooming inside pictures
                                   horizontal_flip = True)#for randomly flipping half of the images horizontally)#initialize train generator
valid_datagen = ImageDataGenerator(rescale = 1.0/255.) #initialize validation generator
test_datagen = ImageDataGenerator(rescale = 1.0/255.)

In [ ]:
train_generator = train_datagen.flow_from_directory('/content/data/train', target_size=(224,224),batch_size=32,class_mode='categorical')
validation_generator = valid_datagen.flow_from_directory('/content/data/val', target_size=(224,224),batch_size=32,class_mode='categorical')
test_generator = test_datagen.flow_from_directory('/content/data/test', target_size=(224,224),batch_size=32,class_mode='categorical',shuffle=False)

In [ ]:
train_generator[0][0].shape

# 2. Load InceptionV3

In [ ]:
import tensorflow as tf

inception = tf.keras.applications.InceptionV3(weights='imagenet',include_top=False,input_shape=(224,224,3))
inception.trainable = False

In [ ]:
inception.summary()

In [ ]:
import keras

# Their Layers
inputs = keras.Input(shape=(224, 224, 3))
x = inception(inputs, training=False)

# Our Layers
x = keras.layers.GlobalAveragePooling2D()(x)
outputs = keras.layers.Dense(31, activation='softmax')(x)

model = keras.Model(inputs, outputs)
model.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
model.summary()

# 3. Train the Model!

Now we'll just train our trainable params (our layers)

In [ ]:
epochs = 15

model.fit(train_generator, epochs=epochs, validation_data=validation_generator)

The performance here isn't great but note that a random guess would give you an accuracy of $1/31 \approx 0.03$! So this model is actually doing really really well.